In [ ]:
# La producción sostenida permite distinguir autores recurrentes de participaciones aisladas.

from pathlib import Path
import json
import sqlite3

import pandas as pd

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATABASE_FILE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# La consulta sigue la relación que convierte publicaciones en evidencia sobre autores.

# authors ← document_authors → documents

database = sqlite3.connect(DATABASE_FILE)

In [ ]:
# La primera CTE limita el análisis al periodo donde se observa la tendencia reciente.

sustained_authors = pd.read_sql_query(
    """
    WITH recent_documents AS (
        SELECT document_id, publication_year
        FROM documents
        WHERE publication_year >= 2020
    ),
    author_production AS (
        SELECT
            a.author_name,
            COUNT(DISTINCT rd.document_id) AS document_count,
            COUNT(DISTINCT rd.publication_year) AS active_year_count,
            MIN(rd.publication_year) AS first_year,
            MAX(rd.publication_year) AS last_year
        FROM authors AS a
        JOIN document_authors AS da ON a.author_id = da.author_id
        JOIN recent_documents AS rd ON da.document_id = rd.document_id
        GROUP BY a.author_id, a.author_name
    )
    SELECT author_name, document_count, active_year_count, first_year, last_year
    FROM author_production
    WHERE active_year_count >= 3
    ORDER BY active_year_count DESC, document_count DESC, author_name
    """,
    database,
)
sustained_authors

In [ ]:
# El resultado se conserva para que otra aplicación pueda usar la misma evidencia.

sustained_authors.to_csv(SUBMISSION_DIR / "sustained_authors.csv", index=False)

questions = [
    {
        "pregunta": "¿Qué autores muestran producción sostenida sobre proptech desde 2020?",
        "archivo_respuesta": "sustained_authors.csv",
    }
]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
database.close()